In [ ]:
import numpy as np
from os import listdir, makedirs
from os.path import exists
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from scipy.interpolate import CubicHermiteSpline, CubicSpline

from threading import Thread


from pendulibrary.integrate import integrate_state
from pendulibrary.plotters import make_gif
from pendulibrary.common_targetters import closest_sol
from pendulibrary.targeter import dc_underconstrained

In [ ]:
def make_gif_one(
    fname,
    frac,
    append_str: str = "",
    frames: int = 120,
    size: float = 4.0,
    dpi: float = 200,
    fps: int = 30,
):
    fname_out = f"../plots/mp4s/{fname}/{append_str}.mp4"
    if exists(fname_out):
        return 0

    data = np.load(f"../database/{fname}.npz")
    x0s = data["x0s"]
    periods = data["periods"]
    Lr, Mr = data["params"]
    # tangents = data["tangents"]
    cols = np.column_stack((x0s, periods))
    steps = np.linalg.norm(np.diff(cols, axis=0), axis=1)
    Ss = np.append(0, np.cumsum(steps))

    # spline_x = CubicHermiteSpline(Ss, x0s, tangents[:, :-1])
    # spline_T = CubicHermiteSpline(Ss, periods, tangents[:, -1])
    spline_x = CubicSpline(Ss, x0s)
    spline_T = CubicSpline(Ss, periods)

    targ = closest_sol(Lr, Mr, 1e-14)

    Xg = spline_x(frac * Ss[-1])
    Tg = spline_T(frac * Ss[-1])

    ind = np.argmin(np.abs(frac * Ss[-1] - Ss))
    # print(x0s.shape,periods[:,None].shape)
    Xs = np.hstack((x0s, periods[:, None]))
    if ind == 0:
        ds = np.linalg.norm(Xs[0] - Xs[1])
    elif ind == Xs.shape[0] - 1:
        ds = np.linalg.norm(Xs[-1] - Xs[-2])
    else:
        ds1 = np.linalg.norm(Xs[ind] - Xs[ind - 1])
        ds2 = np.linalg.norm(Xs[ind + 1] - Xs[ind])
        ds = max(ds1, ds2)

    XX = np.append(Xg, Tg)
    try:
        X, _, _ = dc_underconstrained(XX, targ.g_dg_stm, 1e-5, max_step=ds, max_iter=100)
    except RuntimeError:
        print(f"Error on {fname_out}")
        return 0
    # x0 = X[:4]
    # period = X[-1]
    x0 = X[:-1]
    period = X[-1]

    ts, xs, fs = integrate_state(x0, period, Lr, Mr, 1e-14)
    makedirs(f"../plots/mp4s/{fname}/", exist_ok=True)
    make_gif(xs, ts, fs, Lr, fname_out, frames, size, fps, dpi)


# def make_gif_fam(fam, fracs):
#     for ind, frac in enumerate(fracs):
#         make_gif_one(fam, frac, str(ind), 100)


def make_gif_fam(fam, fracs):
    procs = []
    for ind, frac in enumerate(fracs):
        p = Thread(target=make_gif_one, args=(fam, frac, str(ind), 100))
        procs.append(p)

    for proc in procs:
        proc.start()
    for proc in procs:
        proc.join()

In [ ]:
fams = [f.removesuffix(".npz") for f in listdir("../database")]
inds = np.linspace(1, 0, 10, False)[::-1]

In [ ]:
for fam in tqdm(fams):
    make_gif_fam(fam, inds)

In [ ]:
for fam in tqdm(fams):
    make_gif_fam(fam, inds)